# Lab 3: Data Version Control (DVC) Pipeline Automation

**Course:** ML Operations  
**Objective:** Set up a DVC pipeline, manage data dependencies, and demonstrate reproducibility in the training and evaluation process.

## 3.1 Setup and Imports

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO)

print(f'Working directory: {Path.cwd()}')

## 3.2 Part 1: Project Setup with DVC

### 3.2.1 DVC Installation and Initialization

DVC is installed via `uv add --dev dvc` into the project virtual environment. It integrates with Git to track data and model artifacts alongside code.

In [ ]:
import subprocess
result = subprocess.run(['.venv/bin/dvc', 'version'], capture_output=True, text=True)
print(result.stdout.split('\n')[0])

In [ ]:
# DVC was initialized with: dvc init
# This created the .dvc/ directory and .dvcignore at repo root.
# Verify the DVC directory is present:
dvc_dir = Path('.dvc')
print(f'.dvc/ exists: {dvc_dir.exists()}')
print(f'.dvcignore exists: {Path(".dvcignore").exists()}')
print(f'DVC config: {list(dvc_dir.iterdir())}')

### 3.2.2 Dataset Registry

A dataset registry table stores metadata about each dataset version — source URL, split sizes, random seed. This enables reproducibility auditing: given the `dvc.lock` hash and this registry, any experiment can be fully reproduced.

In [ ]:
import pandas as pd

registry = pd.read_csv('dataset_registry.csv')
print(registry.to_string(index=False))

### 3.2.3 Data Tracking Concept

DVC offers two ways to track data files:

1. **Standalone tracking** (`dvc add data/cifar-10-batches-py`): creates a `.dvc` pointer file committed to git, while the actual data lives in the DVC cache. Suitable for datasets not produced by a pipeline stage.

2. **Pipeline output tracking** (declaring `outs:` in `dvc.yaml`): DVC tracks the file as the output of a stage. On `dvc repro`, the stage is re-run only if its inputs or params have changed. This is the approach used here, since the data is produced by the `download_data` stage.

Using pipeline outputs is preferred for reproducible ML workflows — it makes the data provenance explicit in the pipeline DAG.

In [ ]:
# The data/cifar-10-batches-py directory is tracked as the output of
# the 'download_data' stage in dvc.yaml, not via standalone dvc add.
# data/images is tracked as the output of the 'train' stage.
data_dir = Path('data')
print(f'data/cifar-10-batches-py exists: {(data_dir / "cifar-10-batches-py").exists()}')
print(f'data/images exists: {(data_dir / "images").exists()}')
n_images = len(list((data_dir / 'images').rglob('*.jpg')))
print(f'Total JPEG images extracted: {n_images:,}')

## 3.3 Part 2: Pipeline Creation with DVC

### 3.3.1 Pipeline Definition — dvc.yaml

The pipeline is defined in `dvc.yaml` at the repo root. Each stage specifies:
- `cmd`: the shell command to run
- `deps`: files/directories that, if changed, invalidate the stage cache
- `params`: keys from `params.yaml` whose change triggers a re-run
- `outs`: files/directories the stage produces (tracked by DVC)
- `metrics`: structured JSON output files (enables `dvc metrics show/diff`)

In [ ]:
import yaml

with open('dvc.yaml') as f:
    pipeline = yaml.safe_load(f)

for stage_name, stage in pipeline['stages'].items():
    print(f'\n── Stage: {stage_name}')
    print(f'   cmd  : {stage["cmd"]}')
    print(f'   deps : {stage.get("deps", [])}')
    if 'params' in stage:
        keys = []
        for p in stage['params']:
            if isinstance(p, dict):
                keys.extend(list(p.values())[0])
        print(f'   params: {keys}')
    outs = stage.get('outs', []) + list(stage.get('metrics', {}).keys())
    print(f'   outs : {outs}')

### 3.3.2 Running the Pipeline

`dvc repro` walks the DAG in topological order. For each stage it:
1. Computes MD5 hashes of all `deps` and `params`
2. Compares against the hashes stored in `dvc.lock`
3. Skips the stage if all hashes match; reruns it otherwise

This makes re-runs safe and fast: only changed stages execute.

In [ ]:
# Show what the pipeline would run without actually executing
result = subprocess.run(
    ['.venv/bin/dvc', 'repro', '--dry'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Run the full pipeline (stages already cached after first run — instant)
result = subprocess.run(
    ['.venv/bin/dvc', 'repro'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Visualise the dependency graph
result = subprocess.run(
    ['.venv/bin/dvc', 'dag'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Confirm pipeline is fully up-to-date
result = subprocess.run(
    ['.venv/bin/dvc', 'status'],
    capture_output=True, text=True
)
print(result.stdout or 'Pipeline is up to date.')

## 3.4 Part 3: Parameterization and Configuration

### 3.4.1 Parameter Management with params.yaml

All hyperparameters are stored in `params.yaml`. DVC tracks specific keys per stage:
- `download_data`: no params (URL is hardcoded in the script)
- `train`: all `data.*`, `training.*`, and `output.*` keys
- `evaluate`: `data.*`, `training.batch_size`, `training.num_workers`, `output.save_path`

Changing any tracked key invalidates the corresponding stage and all downstream stages.

In [ ]:
import yaml

with open('params.yaml') as f:
    params = yaml.safe_load(f)

print(yaml.dump(params, default_flow_style=False))

### 3.4.2 Demonstrating Parameter-Change Detection

We temporarily reduce `training.num_epochs` from 20 to 1 to show that DVC detects the change and reruns only the affected stages (`train` and `evaluate`), skipping `download_data` entirely.

In [ ]:
# Record original epochs and change to 1 for a fast demo rerun
with open('params.yaml') as f:
    params = yaml.safe_load(f)

original_epochs = params['training']['num_epochs']
params['training']['num_epochs'] = 1

with open('params.yaml', 'w') as f:
    yaml.dump(params, f, default_flow_style=False)

print(f'Changed num_epochs: {original_epochs} -> 1')

In [ ]:
# Show what DVC detected as changed vs the locked state
result = subprocess.run(
    ['.venv/bin/dvc', 'params', 'diff'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Rerun pipeline — only train + evaluate execute (download_data is skipped)
result = subprocess.run(
    ['.venv/bin/dvc', 'repro'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Show the new metrics after 1-epoch rerun
result = subprocess.run(
    ['.venv/bin/dvc', 'metrics', 'show'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Show metrics diff vs previous run (20-epoch model)
result = subprocess.run(
    ['.venv/bin/dvc', 'metrics', 'diff'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Restore original num_epochs
with open('params.yaml') as f:
    params = yaml.safe_load(f)

params['training']['num_epochs'] = original_epochs

with open('params.yaml', 'w') as f:
    yaml.dump(params, f, default_flow_style=False)

print(f'Restored num_epochs to {original_epochs}')

# Rerun to restore the 20-epoch model in dvc.lock
result = subprocess.run(
    ['.venv/bin/dvc', 'repro'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

## 3.5 Part 4: Code Management and Best Practices

Three pillars of reproducibility in this project:

1. **Configuration in `params.yaml`** — every tunable hyperparameter is declared here. Scripts read from `params.yaml` at runtime; no magic numbers exist in code.

2. **Git + DVC split ownership** — Git tracks code (`scripts/`, `src/`, `dvc.yaml`, `params.yaml`, `dvc.lock`). DVC tracks large binary artifacts (model checkpoints, dataset files). `dvc.lock` is the bridge: it records the exact content hashes of all stage inputs and outputs, committed to git alongside the code.

3. **Python `logging` throughout** — all three scripts (`download_data.py`, `train.py`, `evaluate.py`) use the standard `logging` module to emit stage-level progress messages. This makes pipeline execution auditable without print statements.

In [ ]:
# Show recent commit history including DVC commits
result = subprocess.run(
    ['git', 'log', '--oneline', '-8'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Optional: set up a local DVC remote and push cached artifacts
import os
os.makedirs('/tmp/dvc_remote', exist_ok=True)

result = subprocess.run(
    ['.venv/bin/dvc', 'remote', 'add', '-d', '--force', 'local_remote', '/tmp/dvc_remote'],
    capture_output=True, text=True
)
print('Remote added:', result.returncode == 0)

result = subprocess.run(
    ['.venv/bin/dvc', 'push'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Demonstrate dvc pull (simulates restoring data on a fresh clone)
result = subprocess.run(
    ['.venv/bin/dvc', 'pull', '--dry'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

## 3.6 Part 5: Lab Report

See `docs/lab_report_03.md` for the written report.